## This NoteBook Structure:
1. Load Data
2. Merge Required Tables
3. Convert Date
4. Time-Based Split
5. Missing Value Handling
6. Feature Engineering
7. Encoding
8. Final Dataset
9. Save Processed Data

## 1. Load Data

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
DATA_PATH = Path("../data/raw")

In [8]:
train_df = pd.read_csv(DATA_PATH / "train.csv")
stores_df = pd.read_csv(DATA_PATH / "stores.csv")
holidays_df = pd.read_csv(DATA_PATH / "holidays_events.csv")
oil_df = pd.read_csv(DATA_PATH / "oil.csv")
transactions_df = pd.read_csv(DATA_PATH / "transactions.csv")

In [9]:
## Convert Date Immediatly
train_df["date"] = pd.to_datetime(train_df["date"])
oil_df["date"] = pd.to_datetime(oil_df["date"])
transactions_df["date"] = pd.to_datetime(transactions_df["date"])
holidays_df["date"] = pd.to_datetime(holidays_df["date"])

## 2. Merge Required Tables

In [10]:
# Merge Store Information
df = train_df.merge(
    stores_df,
    on="store_nbr",
    how="left"
)

In [13]:
## Merge Transaction
df = df.merge(
    transactions_df,
    on=["date","store_nbr"],
    how="left"
)

In [17]:
oil_df.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [15]:
## Merge Oil Prices
df = df.merge(
    oil_df,
    on="date",
    how="left"
)

In [16]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,transactions,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN,NaN


In [18]:
## Merge Holidays Information
df = df.merge(
    holidays_df,
    on="date",
    how="left"
)

In [19]:
print(df.shape)
df.head()

(3054348, 17)


,id,date,store_nbr,family,sales,onpromotion,city,state,type_x,cluster,transactions,dcoilwtico,type_y,locale,locale_name,description,transferred
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False


## Sort the DATA

In [20]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type_x,cluster,transactions,dcoilwtico,type_y,locale,locale_name,description,transferred
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False


In [21]:
df = df.sort_values(
    by=["store_nbr","family","date"]
).reset_index(drop=True)

In [22]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type_x,cluster,transactions,dcoilwtico,type_y,locale,locale_name,description,transferred
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,NaN,Holiday,National,Ecuador,Primer dia del ano,False
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,Quito,Pichincha,D,13,2111.0,93.14,NaN,NaN,NaN,NaN,NaN
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,1833.0,92.97,NaN,NaN,NaN,NaN,NaN
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,Quito,Pichincha,D,13,1863.0,93.12,NaN,NaN,NaN,NaN,NaN
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,Quito,Pichincha,D,13,1509.0,NaN,Work Day,National,Ecuador,Recupero puente Navidad,False


In [26]:
df.shape

(3054348, 17)

In [28]:
df.isnull().sum().sort_values(ascending=False)

locale          2551824
type_y          2551824
locale_name     2551824
description     2551824
transferred     2551824
dcoilwtico       955152
transactions     249117
id                    0
date                  0
type_x                0
state                 0
city                  0
onpromotion           0
sales                 0
family                0
store_nbr             0
cluster               0
dtype: int64